# Ensemble guiado: AdaBoost + XGBoost
Roteiro de estudo para a competição Home Credit do GCI. Código simples, sem funções próprias ou classes.

**Objetivo:** prever a probabilidade de TARGET = 1. Vamos usar as 32 variáveis disponíveis, retirar o identificador, comparar dois modelos e calcular uma média das probabilidades.

Este notebook contém exercícios marcados com **TODO**. Complete-os em ordem. As células com `NotImplementedError` param de propósito até você completar o exercício e remover essa linha. Não há resultados de treinamento preenchidos: o notebook ainda não foi executado.

A preparação dos dados está pronta; os passos centrais de aprendizado ficam para você completar. Registre suas conclusões com suas próprias palavras.

## 1. Ambiente e arquivos
No Colab, coloque este notebook na pasta Competition do seu Drive, junto à subpasta input. A célula abaixo instala as bibliotecas se necessário. Se o Colab pedir, reinicie a sessão depois da instalação.

Na célula do caminho, ajuste BASE para a localização real de Competition. Em um Jupyter local, pule a montagem do Drive e use `BASE = Path(".")` se input estiver ao lado do notebook.

In [ ]:
%pip install -q "scikit-learn>=1.2" xgboost pandas numpy

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

SEED = 42
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)

In [ ]:
# Execute esta célula somente no Google Colab.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
BASE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "input" / "train.csv").exists()), Path("/content/drive/MyDrive/Competition"))  # No Colab, ajuste este caminho.
train = pd.read_csv(BASE / "input" / "train.csv")
test = pd.read_csv(BASE / "input" / "test.csv")
sample = pd.read_csv(BASE / "input" / "sample_submission.csv")

print("Treino:", train.shape)
print("Teste:", test.shape)
print("Modelo de envio:", sample.shape)
train.head()

## 2. Entender os dados
A versão lida dos arquivos tem 171.202 linhas de treino, 61.500 de teste e 32 variáveis explicativas. O treino também contém SK_ID_CURR e TARGET; o teste contém SK_ID_CURR.

Aproximadamente 8,07% das linhas de treino têm TARGET = 1. Por isso, usaremos ROC-AUC com probabilidades. Uma acurácia alta, sozinha, pode esconder a dificuldade de identificar a classe minoritária.

**Exercício:** interprete as duas saídas abaixo. Quais colunas têm mais valores ausentes? Por que prever sempre zero daria uma acurácia aparentemente alta?

In [ ]:
print(train["TARGET"].value_counts(normalize=True))
train.isna().mean().sort_values(ascending=False).head(12)

## 3. Separar treino e validação
SK_ID_CURR identifica a linha e será usado apenas para organizar a submissão.
Reservamos 30% do treino para validação, mantendo a proporção das classes com stratify.

Faremos a divisão antes de calcular medianas e definir as colunas das categorias. Assim, a preparação aprende apenas com a parte de treinamento.

In [ ]:
X = train.drop(columns=["SK_ID_CURR", "TARGET"]).copy()
y = train["TARGET"].copy()
X_test = test.drop(columns=["SK_ID_CURR"]).copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_train = X_train.copy()
X_valid = X_valid.copy()

print(X_train.shape, X_valid.shape)

## 4. Preparação simples
Em DAYS_EMPLOYED, o valor 365243 aparece nos dados e equivale a cerca de mil anos. Vamos tratá-lo como ausente. Também representaremos as categorias XNA e XAP como ausência.

Usaremos mediana para números e a categoria "Ausente" para textos. As colunas indicadoras de ausência permitem que os modelos usem essa informação, principalmente em EXT_SOURCE_1 e OWN_CAR_AGE.

Estas são escolhas iniciais de modelagem, não garantias de melhor desempenho. Como usamos árvores, não precisamos padronizar as escalas.

In [ ]:
categoricas = X_train.select_dtypes(exclude="number").columns.tolist()
numericas = X_train.select_dtypes(include="number").columns.tolist()

for dados in [X_train, X_valid, X_test]:
    dados["DAYS_EMPLOYED"] = dados["DAYS_EMPLOYED"].replace(365243, np.nan)
    dados[categoricas] = dados[categoricas].replace(["XNA", "XAP"], np.nan)
    for coluna in numericas:
        dados[coluna + "_AUSENTE"] = dados[coluna].isna().astype(int)
    dados[categoricas] = dados[categoricas].fillna("Ausente")

medianas = X_train[numericas].median().fillna(0)
X_train[numericas] = X_train[numericas].fillna(medianas)
X_valid[numericas] = X_valid[numericas].fillna(medianas)
X_test[numericas] = X_test[numericas].fillna(medianas)

Transformamos categorias em colunas de 0 e 1 com get_dummies. Depois alinhamos validação e teste às colunas de treinamento. Categorias novas ficam sem uma coluna própria, uma limitação simples desta abordagem.

In [ ]:
X_train = pd.get_dummies(X_train, columns=categoricas, dtype=int)
X_valid = pd.get_dummies(X_valid, columns=categoricas, dtype=int)
X_test = pd.get_dummies(X_test, columns=categoricas, dtype=int)

X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

X_train = X_train.astype("float32")
X_valid = X_valid.astype("float32")
X_test = X_test.astype("float32")

print("Colunas após preparação:", X_train.shape[1])
print("Ausentes restantes:", X_train.isna().sum().sum())

## 5. AdaBoost
AdaBoost treina árvores sucessivamente, dando mais atenção aos exemplos classificados incorretamente.

A árvore terá profundidade 2. Vamos começar com 100 árvores e learning_rate de 0,1; são valores iniciais para estudo, sem otimização.

**TODO 1:** treine ada com X_train e y_train usando fit. Em seguida, obtenha as probabilidades de X_valid com predict_proba e selecione a coluna `[:, 1]`. Guarde em `prob_ada`.

Dica: `modelo.fit(entradas, respostas)` treina; `modelo.predict_proba(entradas)[:, 1]` retorna a probabilidade da classe 1.

In [ ]:
ada = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2, random_state=SEED),
    n_estimators=100,
    learning_rate=0.1,
    random_state=SEED
)

# TODO 1: escreva aqui o treinamento e a criação de prob_ada.
raise NotImplementedError("Complete TODO 1 e remova esta linha.")

## 6. XGBoost
XGBoost adiciona árvores para reduzir uma função de erro. Usaremos sua interface XGBClassifier, semelhante à do scikit-learn.

n_estimators define a quantidade de árvores; max_depth limita sua profundidade; learning_rate controla o tamanho de cada passo. tree_method="hist" usa histogramas para construir as árvores de forma eficiente.

**TODO 2:** treine xgb e crie `prob_xgb` com as probabilidades da classe 1 na mesma validação.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=SEED,
    n_jobs=2
)

# TODO 2: escreva aqui o treinamento e a criação de prob_xgb.
raise NotImplementedError("Complete TODO 2 e remova esta linha.")

## 7. Ensemble por média
Os dois modelos já são ensembles de árvores. Agora combinaremos as saídas deles.

Começaremos com pesos iguais: **prob_ensemble = (prob_ada + prob_xgb) / 2**.

**TODO 3:** escreva essa expressão e calcule o ROC-AUC de cada modelo e do ensemble com `roc_auc_score(y_valid, probabilidades)`. Guarde em `auc_ada`, `auc_xgb` e `auc_ensemble`.

Uma média não garante melhora. As probabilidades de AdaBoost e XGBoost podem ter escalas e calibrações diferentes. AUC mede ordenação, não a qualidade da calibração das probabilidades.

In [ ]:
# TODO 3: crie prob_ensemble e os três valores de AUC.
raise NotImplementedError("Complete TODO 3 e remova esta linha.")

In [ ]:
resultados = pd.DataFrame({
    "Modelo": ["AdaBoost", "XGBoost", "Ensemble 50/50"],
    "ROC-AUC validação": [auc_ada, auc_xgb, auc_ensemble]
})
resultados.sort_values("ROC-AUC validação", ascending=False)

**Escreva sua análise:**

- Qual foi o AUC de cada modelo?
- A média melhorou em relação ao melhor modelo individual?
- Qual seria uma explicação para o resultado?
- Que único parâmetro você mudaria no próximo experimento?

Opcional: experimente 25% AdaBoost e 75% XGBoost. Se escolher o peso usando esta validação, o AUC obtido passa a ser uma estimativa usada na seleção; uma avaliação independente ou validação cruzada será necessária para estimar melhor a generalização. Evite escolher pesos por tentativas repetidas no leaderboard.

## 8. Preparar o treino completo
Depois de registrar a comparação, podemos treinar os modelos com todas as linhas rotuladas.

Recalculamos as medianas e as categorias usando o treino completo. Mantemos a mesma estratégia de preparação. Os dados de teste continuam sem participar do cálculo de medianas ou do treinamento.

In [ ]:
X_completo = train.drop(columns=["SK_ID_CURR", "TARGET"]).copy()
X_envio = test.drop(columns=["SK_ID_CURR"]).copy()

for dados in [X_completo, X_envio]:
    dados["DAYS_EMPLOYED"] = dados["DAYS_EMPLOYED"].replace(365243, np.nan)
    dados[categoricas] = dados[categoricas].replace(["XNA", "XAP"], np.nan)
    for coluna in numericas:
        dados[coluna + "_AUSENTE"] = dados[coluna].isna().astype(int)
    dados[categoricas] = dados[categoricas].fillna("Ausente")

medianas_finais = X_completo[numericas].median().fillna(0)
X_completo[numericas] = X_completo[numericas].fillna(medianas_finais)
X_envio[numericas] = X_envio[numericas].fillna(medianas_finais)

X_completo = pd.get_dummies(X_completo, columns=categoricas, dtype=int)
X_envio = pd.get_dummies(X_envio, columns=categoricas, dtype=int)
X_envio = X_envio.reindex(columns=X_completo.columns, fill_value=0)
X_completo = X_completo.astype("float32")
X_envio = X_envio.astype("float32")

**TODO 4:** chame fit novamente para ada e xgb usando X_completo e y. Isso substitui os modelos anteriores por modelos treinados no conjunto completo.

Obtenha `prob_ada_teste` e `prob_xgb_teste` em X_envio. Crie `prob_final` com a mesma média 50/50 avaliada anteriormente. Se você tiver escolhido outro peso de forma explícita, registre-o e use exatamente esse peso.

Após refazer fit, os modelos anteriores de validação foram substituídos. Para repetir a comparação, execute novamente a partir da divisão dos dados.

In [ ]:
# TODO 4: refaça fit, preveja X_envio e crie prob_final.
raise NotImplementedError("Complete TODO 4 e remova esta linha.")

## 9. Salvar no formato da competição
A associação é feita pelo identificador para manter a ordem do arquivo de exemplo.
As verificações abaixo evitam salvar probabilidades ausentes, fora do intervalo ou associadas a IDs incorretos.

Esta célula salva o CSV localmente na pasta output. O envio ao Omnicampus é uma etapa posterior.

In [ ]:
assert test["SK_ID_CURR"].is_unique
assert sample["SK_ID_CURR"].is_unique
assert set(sample["SK_ID_CURR"]) == set(test["SK_ID_CURR"])
assert len(prob_final) == len(test)
assert np.isfinite(prob_final).all()
assert ((prob_final >= 0) & (prob_final <= 1)).all()

previsoes = pd.Series(prob_final, index=test["SK_ID_CURR"])
submission = sample.copy()
submission["TARGET"] = submission["SK_ID_CURR"].map(previsoes)
assert submission["TARGET"].notna().all()

(BASE / "output").mkdir(exist_ok=True)
submission.to_csv(BASE / "output" / "submission_ada_xgb.csv", index=False)
print("Arquivo salvo:", BASE / "output" / "submission_ada_xgb.csv")
submission.head()

## 10. Conclusão e fontes
Complete com suas próprias palavras:
1. O que você entendeu sobre AdaBoost e XGBoost?
2. Por que foi necessário separar validação antes de calcular medianas?
3. O ensemble melhorou? Inclua os valores reais.
4. Quais limitações permanecem?

**Fontes consultadas**
- [Tutorial da competição GCI](https://drive.google.com/file/d/12x-eLfkF39hctLMUqcalWRUnvrlcvCaD/view): organização do fluxo e avaliação por ROC-AUC.
- [README da competição](https://drive.google.com/file/d/1ryBfQlw55rv2CsLXjLIb9BUwQKwzZQ06/view): regras, dados permitidos e formato de envio.
- [Guia de IA e atribuição](https://drive.google.com/file/d/1UonY4qYWCExnQ1nNfqNAxBm9kdbykEAs/view).
- [AdaBoostClassifier — scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html).
- [XGBClassifier — XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html).

Roteiro elaborado com assistência do ChatGPT/Codex em 19/10/2026. Registre suas alterações e a assistência recebida. As regras da competição permitem apenas os dados fornecidos; os demais materiais do curso servem como referência de aprendizado.

**Cobertura da leitura:** os materiais da competição foram consultados anteriormente. A leitura da pasta geral do curso continua parcial; vídeos não foram assistidos e há documentos e notebooks ainda não lidos integralmente. Este roteiro não depende de uma alegação de leitura integral da pasta geral.